In [0]:
from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def list_schema_tables(catalog: str, schema: str):
    """
    List all table names in a Spark schema.

    Args:
        catalog: Catalog name.
        schema: Schema name inside the catalog.

    Returns:
        A list of table names found in the schema.
    """
    return [row.tableName for row in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()]


def map_prefixes_to_tables(table_names: list[str], city_prefixes: list[str]):
    """
    Map each weather prefix to matching table names.

    Matching is performed case-insensitively by checking whether the table
    name starts with the given prefix.

    Args:
        table_names: List of available table names.
        city_prefixes: List of expected weather prefixes, such as `HEL` or `TUR`.

    Returns:
        A dictionary where:
        - key = weather prefix
        - value = list of matching table names
    """
    table_names_upper = [(table_name, table_name.upper()) for table_name in table_names]

    prefix_to_tables = {
        p: [t for (t, u) in table_names_upper if u.startswith(p.upper())]
        for p in city_prefixes
    }

    return prefix_to_tables


def load_weather_table(catalog: str, schema: str, table_name: str, weather_key: str):
    """
    Load and normalize a single raw weather table.

    This transformation:
    - drops precipitation and wind-speed columns
    - converts average temperature to a numeric `temperature` column
    - renames the UTC time column to `hour`
    - casts `hour` to string
    - adds a constant `weather_key` column based on the table prefix

    Args:
        catalog: Catalog name.
        schema: Schema name.
        table_name: Raw weather table name to load.
        weather_key: Prefix assigned to all rows from this table.

    Returns:
        A normalized Spark DataFrame for a single weather table.
    """
    return (
        spark.table(f"{catalog}.{schema}.{table_name}")
        .drop("Tunnin sademäärä [mm]", "Keskituulen nopeus [m/s]")
        .withColumn(
            "temperature",
            F.expr("try_cast(`Lämpötilan keskiarvo [°C]` as double)")
        )
        .withColumnRenamed("Aika [UTC]", "hour")
        .withColumn("hour", F.col("hour").cast("string"))
        .drop("Lämpötilan keskiarvo [°C]")
        .withColumn("weather_key", F.lit(weather_key))
    )


def build_weather_long_dataframe(catalog: str, schema: str, city_prefixes: list[str]):
    """
    Build a unified long-format weather DataFrame from multiple raw tables.

    The function:
    - lists all tables in the source schema
    - matches tables to expected weather prefixes
    - loads each matched table
    - normalizes its schema
    - unions all matched tables into a single DataFrame

    If no table is found for a prefix, a warning is printed and processing
    continues for the remaining prefixes.

    Args:
        catalog: Catalog name.
        schema: Source schema containing raw weather tables.
        city_prefixes: List of expected city/weather prefixes.

    Returns:
        A unified long-format Spark DataFrame containing all matched weather data.

    Raises:
        ValueError: If no tables matched any prefixes.
    """
    table_names = list_schema_tables(catalog, schema)
    prefix_to_tables = map_prefixes_to_tables(table_names, city_prefixes)

    print(prefix_to_tables)

    dataframes: list[DataFrame] = []

    for prefix in city_prefixes:
        matched_tables = prefix_to_tables.get(prefix, [])

        if not matched_tables:
            print(f"WARNING: No table found for prefix {prefix}")
            continue

        for table_name in matched_tables:
            weather_df = load_weather_table(
                catalog=catalog,
                schema=schema,
                table_name=table_name,
                weather_key=prefix,
            )
            dataframes.append(weather_df)

    if not dataframes:
        raise ValueError("No tables matched any prefixes.")

    return reduce(lambda left, right: left.unionByName(right), dataframes)


def write_weather_long_to_bronze(df: DataFrame, destination: str):
    """
    Write the unified long-format weather DataFrame to a bronze table.

    Args:
        df: Spark DataFrame to write.
        destination: Fully qualified destination table name.

    Returns:
        None.

    Side Effects:
        Overwrites the destination table in the metastore.
    """
    df.write.mode("overwrite").saveAsTable(destination)
    print(f"Successfully wrote weather data to {destination}")



if __name__ == "__main__":
    catalog = "fortum_challenge_data"
    schema = "00_landing_weather"

    city_prefixes = ["HEL", "TUR", "TAM", "JOE", "JYV", "KUO", "LAH", "LAP", "OUL", "ROV", "VAA", "HAM", "ESP", "SEI", "VAN", "POR"]

    destination = "fortum_challenge_data.01_bronze.weather_data_long"

    weather_long_df = build_weather_long_dataframe(catalog=catalog, schema=schema, city_prefixes=city_prefixes)

    display(weather_long_df)

    write_weather_long_to_bronze(weather_long_df, destination)
